## Building Multi Agent Systems

# Lesson 8: Hierarchical Multi-Agent Architectures and Orchestrated Task Delegation

In the previous lessons, you mastered the construction of powerful single-agent systems by managing stateful sessions, connecting Model Context Protocol (MCP) servers, and designing native in-process tools. These primitives allow you to engineer highly capable assistants.

However, enterprise software problems are often so multifaceted that a single generalist agent struggles to handle them optimally. Just as industrial engineering teams divide labor among specialists—such as backend developers, security auditors, and technical writers—your AI systems can benefit from the same separation of concerns.

In this lesson, you will learn how to design multi-agent systems where specialized agents coordinate under a central manager to solve multi-layered problems. You will discover how to declare sub-agents with narrow roles, configure a central Orchestrator that delegates subtasks, and trace their concurrent communications using the SDK's internal `Task` tool.

---

## The Sub-Agent Philosophy

A sub-agent is a sandboxed, specialized agent definition configured to excel at a narrow scope of work. Rather than prompting a single agent to handle analysis, code generation, and documentation simultaneously, you break the workflow down into distinct, isolated nodes.

### Why Multi-Agent Hierarchies Excel:

* **Model Optimization:** You can assign specific Claude models to different nodes based on task complexity. For instance, you can use the deep reasoning capabilities of `sonnet` for architectural analysis, while routing heavy text generation tasks to the faster, more cost-effective `haiku` model.
* **Context Preservation:** By limiting the system instructions and toolsets available to each sub-agent, you prevent prompt degradation and keep the model highly focused on its immediate objective.
* **Separation of Concerns:** Debugging and scaling a system becomes significantly simpler when the boundaries between code inspection, patch generation, and test validation are explicitly isolated.

---

## Defining Sub-Agents via `AgentDefinition`

Before initializing a multi-agent cluster, you must define the operational blueprint for each specialized worker node using the `AgentDefinition` class. This class allows you to configure an agent's description, systemic persona instructions, tool access, and model target:

```python
from claude_agent_sdk import AgentDefinition

# 1. Analyzer Agent: Uses Sonnet for deep code audit and analysis
analyzer_agent = AgentDefinition(
    description="Static analysis and security audit specialist.",
    prompt="You are a static code analyzer. Review files for security vulnerabilities, logic bugs, and structural flaws.",
    tools=["Read", "Grep"],
    model="sonnet"
)

```

### `AgentDefinition` Properties Decoded:

* **`description`:** A clear, high-level job title read by the Orchestrator to understand what subtasks this worker is capable of handling.
* **`prompt`:** The specialized system prompt injected into the sub-agent's runtime container.
* **`tools`:** An isolated array of tools this worker is permitted to use. If omitted, the sub-agent inherits all tools available to the master Orchestrator.
* **`model`:** Specifies the model powering the sub-agent (`"haiku"`, `"sonnet"`, or `"opus"`). You can also use `"inherit"` to dynamically match the Orchestrator's model configuration.

### Completing the Specialist Roster: Fixer and Documenter Nodes

To build a fully functional code refactoring cluster, let's declare two additional worker definitions tailored for code generation and technical documentation:

```python
# 2. Fixer Agent: Uses Sonnet to engineer clean, production-ready code patches
fixer_agent = AgentDefinition(
    description="Automated code generation and bug remediation specialist.",
    prompt="You are an automated refactoring engine. Implement high-quality corrections for identified bugs.",
    tools=["Read", "Write"],
    model="sonnet"
)

# 3. Documenter Agent: Uses Haiku for rapid, cost-effective documentation generation
documenter_agent = AgentDefinition(
    description="Technical documentation and compliance writer.",
    prompt="You are a technical writer. Generate comprehensive docstrings and markdown guides.",
    tools=["Read", "Write"],
    model="haiku"
)

```

Notice the strategic model assignment: the `analyzer` and `fixer` require the advanced programming logic of `sonnet`, whereas the `documenter` relies on `haiku` to generate large text outputs quickly and cost-effectively.

---

## Configuring the Orchestrator Registry

With your worker nodes defined, you register them with the main Orchestrator agent using the `agents` parameter in `ClaudeAgentOptions`. This acts as the central registry of specialists available for delegation:

```python
from claude_agent_sdk import ClaudeAgentOptions

options = ClaudeAgentOptions(
    model="haiku", # The Orchestrator primarily manages routing and coordination, handled efficiently by Haiku
    max_turns=15,   # Higher limit to accommodate the multi-agent delegation loops
    allowed_tools=["Read", "Write", "Bash", "Grep"], # Global system-wide tool allowlist
    permission_mode="acceptEdits",
    # Register the sub-agent cluster mapping keys
    agents={
        "analyzer": analyzer_agent,
        "fixer": fixer_agent,
        "documenter": documenter_agent
    }
)

```

### Critical Registry Boundaries:

1. **The Global Tool Allowlist:** The Orchestrator's `allowed_tools` array acts as a master security boundary. **A sub-agent cannot use a tool unless it is explicitly whitelisted here**, even if that tool is declared in the sub-agent's individual `AgentDefinition`.
2. **Orchestrator Model Mapping:** Because coordination, routing, and synthesis don't require heavy coding logic, using `haiku` for the Orchestrator minimizes token overhead while ensuring fast execution.

---

## Defining Delegation Workflows via System Prompts

To make the system functional, you must provide the Orchestrator with explicit orchestration and delegation rules. Its system prompt should act as a manager's handbook, defining exactly when and in what order to call each specialist:

```python
orchestrator_prompt = (
    "You are a Lead Software Architect. Your objective is to refactor code files systematically.\n"
    "When target files are submitted for optimization, you MUST enforce this strict pipeline:\n"
    "1. Delegate to the 'analyzer' sub-agent to extract structural flaws and bugs.\n"
    "2. Pass those analysis notes to the 'fixer' sub-agent to apply targeted patches.\n"
    "3. Route the refactored code to the 'documenter' sub-agent to generate docstrings.\n"
    "Provide a status update back to the user at the conclusion of each stage."
)

# Attach the system prompt to your configuration options
options.system_prompt = orchestrator_prompt

```

---

## Tracing Delegation via the Internal `Task` Tool

When the Orchestrator decides to pass a subtask to a registered specialist, the Claude Agent SDK automatically invokes an internal tool called **`Task`**.

By updating your response display helper to monitor the `Task` tool block, you can gain deep visibility into how the Orchestrator structures its instructions for each worker node:

```python
from claude_agent_sdk import ClaudeSDKClient, AssistantMessage, TextBlock, ToolUseBlock

async def display_multi_agent_stream(client: ClaudeSDKClient):
    """Asynchronously displays streaming text, tool uses, and sub-agent task delegations"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Orchestrator: {block.text}")
                elif isinstance(block, ToolUseBlock):
                    if block.name == "Task":
                        print(f"\n🚀 [DELEGATING TASK TO SUB-AGENT]")
                        if block.input:
                            print(f"  └─► Target Specialist : {block.input.get('subagent_type')}")
                            print(f"  └─► Task Objective    : {block.input.get('description')}")
                            print(f"  └─► Injected Prompt   : {block.input.get('prompt')}")
                    else:
                        print(f"\n🔧 [Running Standard Tool]: {block.name}")

```

---

## Comprehensive Blueprint: Multi-Agent Cluster Execution

Let's combine our sub-agents, orchestrator configurations, and stream monitors into a single, cohesive script to clean up an unoptimized python file (`sample.py`):

```python
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AgentDefinition, tool, create_sdk_mcp_server

# Define your sub-agents
analyzer_agent = AgentDefinition(
    description="Static analysis and security audit specialist.",
    prompt="You are a static code analyzer. Review files for vulnerabilities and structural flaws.",
    tools=["Read", "Grep"],
    model="sonnet"
)

fixer_agent = AgentDefinition(
    description="Automated code generation and bug remediation specialist.",
    prompt="You are an automated refactoring engine. Implement high-quality corrections for identified bugs.",
    tools=["Read", "Write"],
    model="sonnet"
)

documenter_agent = AgentDefinition(
    description="Technical documentation and compliance writer.",
    prompt="You are a technical writer. Generate comprehensive docstrings and markdown guides.",
    tools=["Read", "Write"],
    model="haiku"
)

async def main():
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=30, # High turn ceiling to accommodate multi-agent coordination loops
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent,
        agents={
            "analyzer": analyzer_agent,
            "fixer": fixer_agent,
            "documenter": documenter_agent
        },
        system_prompt=(
            "You are a Lead Software Architect. Your objective is to refactor code files systematically.\n"
            "When target files are submitted for optimization, you MUST enforce this strict pipeline:\n"
            "1. Delegate to the 'analyzer' sub-agent to extract structural flaws and bugs.\n"
            "2. Pass those analysis notes to the 'fixer' sub-agent to apply targeted patches.\n"
            "3. Route the refactored code to the 'documenter' sub-agent to generate docstrings.\n"
            "Provide a status update back to the user at the conclusion of each stage."
        )
    )

    async with ClaudeSDKClient(options=options) as client:
        # Initiate the workflow with a high-level command
        await client.query("Please analyze and completely optimize the sample.py file.")
        await display_multi_agent_stream(client)

if __name__ == "__main__":
    anyio.run(main)

```

---

## Multi-Turn Trace Analysis

When running the automation script above, watch how the Orchestrator steps through the pipeline sequentially, preparing data and invoking specialists via the `Task` tool:

### Phase 1: Context Preparation & Analysis Delegation

The Orchestrator initializes the workflow by locating the target file, analyzing its contents, and then dispatching it to the `analyzer` sub-agent:

```text
💬 Orchestrator: I'll help you optimize the sample.py file. Let me inspect its layout first.
🔧 [Running Standard Tool]: Read

💬 Orchestrator: File loaded. Beginning Phase 1 by spinning up our static code analyzer node.
🚀 [DELEGATING TASK TO SUB-AGENT]
  └─► Target Specialist : analyzer
  └─► Task Objective    : Analyze sample.py for structural and style flaws.
  └─► Injected Prompt   : Audit the code at sample.py. Check for division by zero errors, missing input validation, and inefficient iteration loops. Provide a structured bug report.

```

### Phase 2: Remediating Issues

The `analyzer` finishes its inspection loop and passes a structured report back to the Orchestrator. The Orchestrator processes the findings and forwards them directly to the `fixer` node:

```text
💬 Orchestrator: Phase 1 complete. The analyzer identified critical bugs, including a potential ZeroDivisionError and a missing summation optimization. Spawning the refactoring engine to apply patches.

🚀 [DELEGATING TASK TO SUB-AGENT]
  └─► Target Specialist : fixer
  └─► Task Objective    : Remediate bugs discovered in sample.py.
  └─► Injected Prompt   : Apply the following modifications to sample.py:
                          1. Protect against empty lists to avoid ZeroDivisionError.
                          2. Replace the manual iteration loop with Python's built-in sum() function.
                          Write the refactored, production-ready code back to the file.

```

### Phase 3: Enhancing Documentation

The `fixer` patches the codebase via the `Write` tool. The Orchestrator reviews the updated file and hands it off to the `documenter` sub-agent to add clean docstrings:

```text
💬 Orchestrator: Phase 2 complete. The codebase is now structurally stable. Initializing the documentation pipeline to add type hints and docstrings.

🚀 [DELEGATING TASK TO SUB-AGENT]
  └─► Target Specialist : documenter
  └─► Task Objective    : Add comprehensive documentation layer to sample.py.
  └─► Injected Prompt   : Read the patched code at sample.py. Append module-level documentation and complete docstrings matching standard PEP 8 styling conventions.

```

### Phase 4: Final Verification

Once the `documenter` updates the file, the Orchestrator performs a final inspection pass and outputs a clean technical summary:

```text
🔧 [Running Standard Tool]: Read

💬 Orchestrator: 🎉 Reorganization and Refactoring Complete!
- Step 1 (Analysis): Completed. Identified 3 core structural vulnerabilities.
- Step 2 (Remediation): Completed. Replaced loops with sum() and added division-by-zero safeguards.
- Step 3 (Documentation): Completed. Added full type hints and PEP 8 docstrings.
The file sample.py is now fully production-ready! 🚀

```

---

## Scaling Sub-Agents with Custom MCP Tools

The system built above coordinates workers using built-in filesystem tools. However, you can also give your sub-agents access to advanced custom tools or external Model Context Protocol (MCP) servers—for example, giving a research agent access to specialized documentation search APIs.

When attaching custom protocol engines to sub-agent definitions, remember the core security gatekeeping rule: **The master Orchestrator's `allowed_tools` list must explicitly include any namespaced MCP tools that sub-agents intend to use.**

```python
# Declare an external documentation lookup MCP server process configuration
mcp_doc_config = {
    "type": "stdio",
    "command": "npx",
    "args": ["-y", "@upstash/context7-mcp"]
}

# Construct a specialized researcher sub-agent definition
researcher_agent = AgentDefinition(
    description="Documentation researcher and API compliance specialist.",
    prompt="Search external library schemas to verify production best practices.",
    tools=[
        "mcp__docs__resolve-library-id",
        "mcp__docs__get-library-docs",
        "Read"
    ],
    model="haiku"
)

options = ClaudeAgentOptions(
    model="haiku",
    mcp_servers={"docs": mcp_doc_config}, # Mount the server process
    allowed_tools=[
        "Read", "Write", "Grep", "Bash",
        # The Orchestrator must explicitly allow these keys so the researcher can access them!
        "mcp__docs__resolve-library-id",
        "mcp__docs__get-library-docs"
    ],
    agents={
        "analyzer": analyzer_agent,
        "researcher": researcher_agent # Mount the sub-agent into the cluster
    }
)

```

---

## Summary

You have mastered how to design and orchestrate complex, hierarchical multi-agent clusters:

* **`AgentDefinition`** encapsulates isolated worker nodes with specialized system prompts, tool rules, and model selections.
* The **`agents` dictionary** registers your specialists within the Orchestrator's workspace option space.
* The **`Task` tool** handles the internal communication channel between the central manager and individual worker processes.
* The Orchestrator's **`allowed_tools` register** acts as the primary system firewall, meaning it must explicitly include all built-in and external MCP tools utilized across the entire sub-agent fleet.

## Creating Your First Agent Specialist

Before diving into complex multi-agent orchestration, let's start by understanding the fundamental building block: the AgentDefinition class itself.

Your task is to create a single AgentDefinition for an analyzer agent. This agent will be a code analysis specialist designed to review code for issues, patterns, and improvements. Configure it with four key parameters:

    Set the description to identify its role.
    Write a prompt that defines how it should approach its work.
    Give it a tools list containing "Read" and "Grep".
    Set the model to "sonnet" for deep reasoning capabilities.

The print statements are already in place to display the agent's properties once you create it. Master this foundation, and you'll be ready to build complete multi-agent systems in the exercises ahead!

```
from claude_agent_sdk import AgentDefinition

# TODO: Create an AgentDefinition for the analyzer agent
analyzer_agent = 

# Print the agent's properties to understand what an AgentDefinition contains
print(f"Description: {analyzer_agent.description}")
print(f"Prompt: {analyzer_agent.prompt}")
print(f"Tools: {analyzer_agent.tools}")
print(f"Model: {analyzer_agent.model}")
```

Berikut adalah kode program lengkap yang telah diisi untuk membuat objek `AgentDefinition` bagi agen spesialis analis kode (`analyzer_agent`).

Konfigurasi ini menetapkan deskripsi peran, instruksi sistem internal (*system prompt*), batasan alat eksekusi, serta model Claude yang disesuaikan dengan kebutuhan penalaran mendalam:

```python
from claude_agent_sdk import AgentDefinition

# Create an AgentDefinition for the analyzer agent
analyzer_agent = AgentDefinition(
    description="Code analysis specialist",
    prompt="You are a code analyzer. Review code for issues, patterns, and improvements systematically.",
    tools=["Read", "Grep"],
    model="sonnet"
)

# Print the agent's properties to understand what an AgentDefinition contains
print(f"Description: {analyzer_agent.description}")
print(f"Prompt: {analyzer_agent.prompt}")
print(f"Tools: {analyzer_agent.tools}")
print(f"Model: {analyzer_agent.model}")

```

### Penjelasan Parameter Fondasi:

1. **`description`:** Bertindak sebagai kartu identitas atau nama jabatan spesialis. Deskripsi ini sangat penting karena dibaca oleh agen Orkestrator utama untuk menentukan kapan subtask inspeksi kode harus dilemparkan ke agen analis ini.
2. **`prompt`:** Mengisolasi perilaku instruksi sistem agar model fokus sepenuhnya pada audit kualitas statis tanpa terdistraksi oleh tugas penulisan kode draf.
3. **`tools`:** Membatasi akses alat agar lebih hemat token dan aman. Di sini, agen hanya diberi perkakas pembaca (`Read`) dan pencari pola teks (`Grep`).
4. **`model="sonnet"`:** Memilih model dengan kapasitas penalaran (*reasoning*) tertinggi untuk membedah arsitektur kode dan celah logika yang rumit.

## Delegating Analysis to Your Specialist

Great work learning how to define individual agent specialists! Now it's time to see the real power of multi-agent systems by connecting your analyzer agent to the orchestrator.

Currently, the code has an analyzer agent defined, but the orchestrator doesn't know it exists. Without registering the agent and providing delegation instructions, Claude will try to do all the work itself — analysis, fixes, and documentation — without leveraging the specialized analyzer you've created.

Your task is to transform this into a proper multi-agent workflow:

    Register the analyzer_agent in the agents parameter of ClaudeAgentOptions using the key "analyzer".
    Update the system_prompt to instruct the orchestrator to first delegate to the analyzer for finding issues, then handle the fixes and documentation itself.

Once you make these changes, you'll see the orchestrator delegate the analysis phase to your specialist while handling the remaining work on its own — your first taste of agent coordination in action!

```
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # TODO: Register the analyzer_agent in the agents parameter and modify the system_prompt
    # to instruct Claude to first delegate to 'analyzer' for analysis, then handle fixes and documentation itself
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        system_prompt="You are a Lead Developer. Your goal is to improve code quality."
    )

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```
Berikut adalah kode program lengkap yang telah disesuaikan untuk mendaftarkan sub-agen spesalis analis (`analyzer_agent`) ke dalam konfigurasi `ClaudeAgentOptions` serta memperbarui instruksi pendelegasian pada *system prompt*:

```python
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # Register the analyzer_agent in the agents parameter and modify the system_prompt
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        # 1. Daftarkan spesialis ke dalam registri agen orkestrator
        agents={
            "analyzer": analyzer_agent
        },
        # 2. Perbarui instruksi pendelegasian alur kerja (workflow orchestration)
        system_prompt=(
            "You are a Lead Developer. Your goal is to improve code quality.\n"
            "When asked to improve a file, you MUST follow this structural workflow:\n"
            "1. Delegate the initial code inspection to the 'analyzer' sub-agent to find issues.\n"
            "2. Once you receive the analyzer's report, resolve the identified issues yourself using your tools.\n"
            "3. Finally, write comprehensive documentation for the improved code yourself."
        )
    )

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Analisis Arsitektur Mekanisme Pendelegasian:

1. **Registri Kunci (`agents={"analyzer": ...}`)**: Menetapkan pengidentifikasi string `"analyzer"` ke objek definisi agen. String kunci inilah yang dibaca oleh model induk (Orkestrator) saat memutuskan ke mana ia harus mengirimkan subtask.
2. **Alur Kerja Preskriptif**: Penyusunan kalimat *"you MUST follow this structural workflow"* pada `system_prompt` memaksa model induk bertindak sebagai manajer proyek. Alih-alih langsung memodifikasi berkas `sample.py` secara gegabah menggunakan perkakas `Write`, model induk (`haiku`) akan memanggil perkakas internal SDK bernama `Task` untuk memicu agen spesialis (`sonnet`) melakukan audit mendalam terlebih dahulu.

## Observing Agent Delegation in Action

Excellent progress on connecting your analyzer specialist to the orchestrator! Now that you have a working multi-agent system with proper delegation, it's time to peek under the hood and see exactly how the orchestrator communicates with its specialists.

Right now, when you run the code, you'll see tool calls like 🔧 [Tool: Task] appear in the output, but you won't see what instructions the orchestrator is actually sending to the analyzer. The Task tool is the internal mechanism the SDK uses for delegation, and its input parameter contains valuable information: which specialist is being called (subagent_type), a brief summary of the task (description), and the detailed instructions (prompt) the orchestrator crafted for that specialist.

Your task is to enhance the display_response helper function in utils.py to reveal these delegation details. After the line that prints the tool name, add logic to:

    Check if the tool being used is named "Task"
    If it is, check whether block.input exists
    Loop through the key-value pairs in block.input and print each one

Once you make this change, you'll be able to observe the orchestrator's delegation strategy in action, seeing exactly how it translates your high-level query into specific instructions for the analyzer specialist!

```
from claude_agent_sdk import (
    ClaudeSDKClient,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    # TODO: If this tool is 'Task', print its input parameters to observe delegation

# main.py
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # Configure the main agent (The Orchestrator)
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        agents={
            "analyzer": analyzer_agent
        },
        system_prompt=(
            "You are a Lead Developer. Your goal is to improve code quality.\n"
            "When asked to improve a file, you MUST follow this strict process:\n"
            "1. Delegate to 'analyzer' to find issues\n"
            "2. Fix the issues yourself using your tools\n"
            "3. Add documentation yourself using your tools\n"
            "Report back after each step."
        )
    )

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is the complete implementation for your task. The `utils.py` file has been enhanced with conditional check logic under the `ToolUseBlock` to unpack and print the dictionary payload of the internal `Task` delegation tool.

### 1. Updated `utils.py`

```python
from claude_agent_sdk import (
    ClaudeSDKClient,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    
                    # Check if the tool being used is named "Task"
                    if block.name == "Task":
                        # Check whether block.input parameters exist
                        if block.input:
                            # Loop through the key-value pairs in block.input and print each one
                            for key, value in block.input.items():
                                print(f"  └─► {key}: {value}")

```

### 2. Complete Runnable Example (`main.py`)

You can use the `main.py` below alongside the updated helper to execute and trace the multi-agent system:

```python
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # Configure the main agent (The Orchestrator)
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        agents={
            "analyzer": analyzer_agent
        },
        system_prompt=(
            "You are a Lead Developer. Your goal is to improve code quality.\n"
            "When asked to improve a file, you MUST follow this strict process:\n"
            "1. Delegate to 'analyzer' to find issues\n"
            "2. Fix the issues yourself using your tools\n"
            "3. Add documentation yourself using your tools\n"
            "Report back after each step."
        )
    )

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Expected Output Trace

When running this code, the terminal stream will explicitly reveal the orchestrator's step-by-step delegation strategy instead of masking it behind a generic `[Tool: Task]` log block:

```text
💬 Claude Response:
I will help you improve the sample.py file. First, let's analyze the file for any underlying issues by delegating it to our specialized code analyzer.

🔧 [Tool: Task]
  └─► subagent_type: analyzer
  └─► description: Analyze sample.py for potential improvements, style issues, and performance gaps.
  └─► prompt: Review the contents of sample.py. Identify error-handling vulnerabilities, edge cases, performance bugs, and best-practice violations. Generate a clear report.

```

## Building Your Code Review Team

Now that you've seen how a small multi-agent system coordinates specialists to solve complex problems, it's time to build a larger team by completing the missing pieces of the code review workflow.

Your job is to create the two remaining specialists and integrate them into the orchestration system. Here's what you need to do:

    Define the fixer_agent using AgentDefinition with an appropriate description, prompt, and tools
    Define the documenter_agent using AgentDefinition with an appropriate description, prompt, and tools
        Both the fixer and documenter need Read and Write tools to examine and modify code
        Choose an appropriate model for each agent based on the complexity of their tasks
        Each agent's prompt should clearly define its specific role in the workflow
    Register both new agents in the orchestrator's agents dictionary with the keys "fixer" and "documenter"
    Modify the orchestrator's system_prompt by updating steps 2 and 3 to delegate work to the fixer and documenter

Once you complete these steps, your multi-agent system will coordinate all three specialists to transform problematic code into production-ready code!

```
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # 1. Define custom sub-agents individually
    
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # TODO: Define the fixer_agent that fixes bugs
    # Hint: Its prompt should focus on identifying and fixing code issues

    # TODO: Define the documenter_agent that writes documentation
    # Hint: Its prompt should focus on creating clear, comprehensive documentation

    # 2. Configure the main agent (The Orchestrator)
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        # TODO: Add the fixer and documenter agents to the agents dictionary
        agents={
            "analyzer": analyzer_agent,
        },
        # TODO: Update steps 2 and 3 to delegate to 'fixer' and 'documenter' agents
        system_prompt=(
            "You are a Lead Developer. Your goal is to improve code quality.\n"
            "When asked to improve a file, you MUST follow this strict process:\n"
            "1. Analyze the file for issues\n"
            "2. Fix the issues yourself using your tools\n"
            "3. Add documentation yourself using your tools\n"
            "Report back after each step."
        )
    )

    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

# sample.py
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    return total / len(numbers)

def process_data(data):
    # TODO: Add error handling
    result = calculate_average(data)
    return result
```

Here is the complete solution code that builds the full hierarchical code review team.

The `fixer_agent` is configured with `sonnet` to utilize its deep semantic understanding for generating high-quality architectural patches. Meanwhile, the `documenter_agent` is configured with `haiku` to cost-effectively write verbose technical documentation and docstrings.

```python
import anyio
from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AgentDefinition,
)
from utils import display_response


async def main():
    # 1. Define custom sub-agents individually
    
    # Analyzer Agent: Uses Sonnet for deep reasoning and Read/Grep tools to explore the codebase
    analyzer_agent = AgentDefinition(
        description="Code analysis specialist",
        prompt="You are a code analyzer. Review code for issues, patterns, and improvements.",
        tools=["Read", "Grep"],
        model="sonnet"
    )

    # Define the fixer_agent that fixes bugs using Sonnet for intelligent code changes
    fixer_agent = AgentDefinition(
        description="Bug fixing specialist",
        prompt="You are a bug fixer. Read the code analyzer's findings and implement robust corrections and fixes efficiently.",
        tools=["Read", "Write"],
        model="sonnet"
    )

    # Define the documenter_agent that writes documentation using Haiku for speed and cost-efficiency
    documenter_agent = AgentDefinition(
        description="Documentation specialist",
        prompt="You are a documentation writer. Review the fixed code and create clear, comprehensive docstrings and inline comments.",
        tools=["Read", "Write"],
        model="haiku"
    )

    # 2. Configure the main agent (The Orchestrator)
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=30,  # Increased from 15 to 30 to comfortably support multiple cascading sub-agent turns
        allowed_tools=["Read", "Write", "Bash", "Grep"],
        permission_mode="acceptEdits",
        # Add the fixer and documenter agents to the agents dictionary registry
        agents={
            "analyzer": analyzer_agent,
            "fixer": fixer_agent,
            "documenter": documenter_agent
        },
        # Update steps 2 and 3 to delegate tasks instead of forcing the orchestrator to do the work
        system_prompt=(
            "You are a Lead Developer. Your goal is to improve code quality.\n"
            "When asked to improve a file, you MUST follow this strict process:\n"
            "1. Delegate to 'analyzer' to find issues\n"
            "2. Delegate to 'fixer' to resolve the identified issues and bugs\n"
            "3. Delegate to 'documenter' to add comprehensive documentation\n"
            "Report back after each step."
        )
    )

    # 3. Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Please improve the sample.py file.")

        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Architectural Highlights of Your Pipeline

* **Strategic Model Split:** By separating logic generation (`sonnet` for `analyzer`/`fixer`) from language text elaboration (`haiku` for `documenter`), you maximize code correctness while significantly reducing input/output token costs.
* **`max_turns=30` Adjustment:** Because three separate specialized sub-agents are spinning up sequentially—each running through their own internal tool execution loops via the SDK's internal `Task` tool—the execution threshold has been scaled up to prevent early turn cut-offs.
* **The Global Allowlist Gatekeeper:** Notice that even though the sub-agents specify `["Read", "Write"]` or `["Read", "Grep"]`, they are bound by the Orchestrator's master `allowed_tools` boundaries. If a tool name is omitted from the Orchestrator's list, no nested child agent can call it.

Based on our multi-agent configuration, the orchestrator delegates the job of tracking down and resolving that specific `# TODO: Add error handling` comment to the specialized **fixer** sub-agent.

Here is how the code in `sample.py` is transformed from its original unoptimized state into a robust, production-ready implementation once the multi-agent refactoring pipeline runs.

### 1. Original `sample.py` (Before Refactoring)

This is what the file looks like initially, containing raw logic bugs, a manual loop, missing type safety, and the outstanding `# TODO` comment:

```python
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    return total / len(numbers)

def process_data(data):
    # TODO: Add error handling
    result = calculate_average(data)
    return result

```

---

### 2. Refactored `sample.py` (After Multi-Agent Execution)

Once the orchestrator passes the task to the **analyzer** (to map out the edge cases), the **fixer** (to resolve the bugs and the TODO), and the **documenter** (to add PEP 8 and NumPy-style docstrings), the final code written back to the workspace becomes completely robust:

```python
"""
Data Processing and Statistical Utilities Module.

This module provides enterprise-level mathematical utilities for calculating 
averages and managing data arrays safely with full error handling and type validation.
"""

from typing import List, Union

def calculate_average(numbers: List[Union[int, float]]) -> float:
    """
    Calculate the arithmetic mean of a list of numeric values.

    Parameters:
    numbers (List[int, float]): A list containing integers or floats.

    Returns:
    float: The calculated average value.

    Raises:
    ValueError: If the provided list is empty or contains non-numeric data.
    """
    # Defensive programming: protection against empty lists (ZeroDivisionError)
    if not numbers:
        raise ValueError("Calculation failed: The numbers list cannot be empty.")
    
    # Input validation: ensure everything inside the list is an int or float
    if not all(isinstance(num, (int, float)) for num in numbers) or any(isinstance(num, bool) for num in numbers):
        raise ValueError("Calculation failed: All elements in the list must be numeric.")

    # Optimization: Replaced inefficient manual iteration loop with built-in sum()
    return sum(numbers) / len(numbers)


def process_data(data: List[Union[int, float]]) -> dict:
    """
    Process raw data streams and extract meaningful statistical metrics.

    This function resolves the previous error handling tracking gaps by safely
    intercepting calculation faults and formatting them into structured metrics.

    Parameters:
    data (List[int, float]): Raw input numerical list.

    Returns:
    dict: A summary containing status information and calculation metrics.
    """
    # RESOLVED TODO: Comprehensive error handling block implemented
    try:
        # Validate data types at entry point
        if not isinstance(data, list):
            return {
                "status": "error",
                "message": f"Invalid input type: Expected list, got {type(data).__name__}."
            }
            
        average_result = calculate_average(data)
        
        return {
            "status": "success",
            "metrics": {
                "average": average_result,
                "data_points_processed": len(data)
            }
        }
        
    except ValueError as val_error:
        # Catch and handle specific numeric or empty list calculation errors
        return {
            "status": "error",
            "message": str(val_error)
        }
    except Exception as general_error:
        # Global fallback protection for unexpected runtime anomalies
        return {
            "status": "error",
            "message": f"An unexpected system anomaly occurred: {str(general_error)}"
        }

```

---

### What Key Optimizations Were Made to Resolve the TODO?

* **Zero-Division Protection:** The original code would instantly crash with a `ZeroDivisionError` if `numbers` was empty because `len(numbers)` would be `0`. The refactored version checks for an empty list upfront and raises a clear `ValueError`.
* **Type Safety Enforcement:** Python allows dynamic typing, meaning a user could pass a list of strings (`["a", "b"]`) or a mixed boolean array, which would cause an unhandled runtime error. The optimized function explicitly filters out non-numeric entries and handles them gracefully.
* **Enhanced `process_data` Return Schema:** Instead of blindly returning a raw result that could fail silently or crash the upstream service, the TODO fix introduces a structured `try-except` block. It translates failures into safe, descriptive status payloads (`{"status": "error", "message": "..."}`) which are production-ready.
* **Algorithmic Efficiency:** The slow manual iteration loop (`for num in numbers: total = total + num`) was replaced with Python's optimized, highly efficient built-in C-level `sum()` function.

## Delegating MCP Tools to Specialists

Excellent work building complete multi-agent teams with specialized roles! Now it's time to see how MCP tools fit into the multi-agent architecture you've mastered.

The code is currently set up as a single-agent system: it connects to the Context7 MCP server and lists the MCP tools in the allowed_tools. However, the main agent itself is using these tools directly, rather than delegating documentation research to a specialist agent.

Your task is to refactor this into a proper multi-agent system by creating a documentation research specialist:

    Define a researcher_agent using AgentDefinition with an appropriate description and prompt for documentation research.
    Give it the two MCP tools in its tools list: "mcp__context7__resolve-library-id" and "mcp__context7__get-library-docs".
    Register the researcher in the orchestrator's agents parameter with the key "researcher".
    Add a system_prompt to the orchestrator instructing it to delegate documentation research tasks to the researcher specialist.

Once you complete these changes, you'll see how MCP tools flow through a multi-agent hierarchy: the orchestrator maintains the global allow-list, but delegates the actual research work to the specialist!

```
import anyio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AgentDefinition
)
from utils import display_response


async def main():
    # Define external MCP server configuration
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # TODO: Create a researcher_agent using AgentDefinition
    # - Give it a description identifying it as a documentation research specialist
    # - Write a prompt explaining its role is to search library documentation
    # - Provide it with the MCP tools: "mcp__context7__resolve-library-id" and "mcp__context7__get-library-docs"

    # TODO: Configure the orchestrator below:
    # - Register the researcher_agent in an 'agents' parameter with key "researcher"
    # - Add a 'system_prompt' that instructs the orchestrator to delegate documentation research to the researcher
    # - Keep the MCP tools in 'allowed_tools' - this is required for sub-agents to use them
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=10,
        # Connect the external server
        mcp_servers={
            "context7": context7_config
        },
        # MCP tools must be in orchestrator's allowed_tools for sub-agents to use them
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ]
    )
    
    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "Search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

The error `ModuleNotFoundError: No module named 'utils'` occurs because the Python environment running your `main.py` script cannot locate the `utils.py` file in its search path. In this SDK environment, `main.py` looks for `utils.py` in the same directory to import the `display_response` helper function.

Here are the solutions to fix this issue depending on how you are running the code:

### Solution 1: Inline the `display_response` Function (Recommended)

If you are running this within an exercise container or sandbox where you cannot easily verify or create separate files, the cleanest approach is to remove the `import` statement entirely and define the `display_response` helper function directly inside `main.py`.

Here is the fully self-contained, consolidated version of your **MCP Tools Delegation** exercise code:

```python
import anyio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AgentDefinition,
    AssistantMessage,
    TextBlock,
    ToolUseBlock
)

# =====================================================================
# INLINED TELEMETRY HELPER (Bypasses the 'utils' import)
# =====================================================================
async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with full task delegation tracing"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    # Track and expose internal sub-agent task delegations
                    if block.name == "Task" and block.input:
                        for key, value in block.input.items():
                            print(f"  └─► {key}: {value}")

# =====================================================================
# MAIN RUNTIME ENGINE
# =====================================================================
async def main():
    # Define external MCP server configuration
    context7_config = {
        "type": "stdio",  # Communicates via stdin/stdout
        "command": "npx",
        "args": ["-y", "@upstash/context7-mcp"]
    }

    # 1. Define the researcher_agent specifically for documentation research
    researcher_agent = AgentDefinition(
        description="Documentation research specialist",
        prompt=(
            "You are a technical documentation researcher. Your single focus is to search "
            "external library documentation, resolve library IDs, and retrieve comprehensive "
            "SDK documentation using your provided MCP tools to answer research queries accurately."
        ),
        tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ],
        model="haiku"  # Haiku is fast, accurate, and cost-effective for documentation lookups
    )

    # 2. Configure the main agent (The Orchestrator)
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,  # Increased to allow adequate turns for multi-agent loops
        mcp_servers={
            "context7": context7_config
        },
        agents={
            "researcher": researcher_agent
        },
        allowed_tools=[
            "mcp__context7__resolve-library-id",
            "mcp__context7__get-library-docs"
        ],
        system_prompt=(
            "You are a Lead Project Orchestrator. Your goal is to coordinate technical research.\n"
            "When asked to search for documentation or look up library functionalities, you MUST follow this process:\n"
            "1. Delegate the task directly to the 'researcher' sub-agent to retrieve authoritative documentation.\n"
            "2. Once the researcher reports back, analyze its findings and synthesize a clear summary for the user."
        )
    )
    
    # 3. Run the agent execution context
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "Search for documentation on how the "
            "Claude Agent SDK (Python) uses MCP servers."
        )
        
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

---

### Solution 2: Fix the Workspace Pathing

If you prefer keeping the files separate, ensure your directory layout matches this structure:

```text
/usercode/FILESYSTEM/
├── main.py
└── utils.py

```

If you are invoking Python from outside the directory where these files sit, it won't resolve the local path. Make sure you explicitly switch directories in your shell workspace before execution:

```bash
cd /usercode/FILESYSTEM/
python main.py

```